# 01 — Data Audit

Confirm schemas, observation levels, keys, date coverage, and incomplete periods before cleaning or feature derivation.

In [0]:
from pyspark.sql import functions as F

RAW_PATH = "/Volumes/workspace/olist/raw"

In [0]:
customers = spark.read.csv(f"{RAW_PATH}/olist_customers_dataset.csv", header=True, inferSchema=True)
geo_locations = spark.read.csv(f"{RAW_PATH}/olist_geolocation_dataset.csv", header=True, inferSchema=True)
orders = spark.read.csv(f"{RAW_PATH}/olist_orders_dataset.csv", header=True, inferSchema=True)
order_items = spark.read.csv(f"{RAW_PATH}/olist_order_items_dataset.csv", header=True, inferSchema=True)
order_payments = spark.read.csv(f"{RAW_PATH}/olist_order_payments_dataset.csv", header=True, inferSchema=True)
order_reviews = spark.read.csv(f"{RAW_PATH}/olist_order_reviews_dataset.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{RAW_PATH}/olist_products_dataset.csv", header=True, inferSchema=True)
sellers = spark.read.csv(f"{RAW_PATH}/olist_sellers_dataset.csv", header=True, inferSchema=True)

## Schema inspection

In [0]:
orders.printSchema()
order_items.printSchema()
order_payments.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



## Grain and key checks

In [0]:
display(orders.agg(F.count("*").alias("rows"), F.countDistinct("order_id").alias("unique_order_ids")))
display(order_items.agg(F.count("*").alias("rows"), F.countDistinct("order_id", "order_item_id").alias("unique_order_item_keys")))
display(order_payments.agg(F.count("*").alias("rows"), F.countDistinct("order_id", "payment_sequential").alias("unique_payment_keys")))

rows,unique_order_ids
99441,99441


rows,unique_order_item_keys
112650,112650


rows,unique_payment_keys
103886,103886


## Monthly coverage audit

The last calendar month may be incomplete. It must not automatically be interpreted as a business decline.

In [0]:
order_time_audit = (
    orders
    .withColumn("purchase_month", F.to_date(F.date_trunc("month", "order_purchase_timestamp")))
    .groupBy("purchase_month")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum(F.when(F.col("order_status").isin("canceled", "unavailable"), 1).otherwise(0)).alias("canceled_or_unavailable"),
        F.min("order_purchase_timestamp").alias("first_purchase"),
        F.max("order_purchase_timestamp").alias("last_purchase")
    )
    .orderBy("purchase_month")
)

display(order_time_audit)

purchase_month,order_count,canceled_or_unavailable,first_purchase,last_purchase
2016-09-01,4,2,2016-09-04T21:15:19.000Z,2016-09-15T12:16:38.000Z
2016-10-01,324,31,2016-10-02T22:07:52.000Z,2016-10-22T08:25:27.000Z
2016-12-01,1,0,2016-12-23T23:16:47.000Z,2016-12-23T23:16:47.000Z
2017-01-01,800,13,2017-01-05T11:56:06.000Z,2017-01-31T23:37:58.000Z
2017-02-01,1780,62,2017-02-01T00:04:17.000Z,2017-02-28T23:47:08.000Z
2017-03-01,2682,65,2017-03-01T00:01:30.000Z,2017-03-31T23:54:45.000Z
2017-04-01,2404,27,2017-04-01T00:54:10.000Z,2017-04-30T23:48:13.000Z
2017-05-01,3700,60,2017-05-01T01:18:22.000Z,2017-05-31T22:59:52.000Z
2017-06-01,3245,40,2017-06-01T00:05:38.000Z,2017-06-30T23:20:08.000Z
2017-07-01,4026,80,2017-07-01T00:04:15.000Z,2017-07-31T23:55:27.000Z


In [0]:
last_months = order_time_audit.orderBy(F.desc("purchase_month")).limit(8)
display(last_months.orderBy("purchase_month"))

purchase_month,order_count,canceled_or_unavailable,first_purchase,last_purchase
2018-03-01,7211,43,2018-03-01T00:00:00.000Z,2018-03-31T23:54:10.000Z
2018-04-01,6939,20,2018-04-01T00:11:32.000Z,2018-04-30T23:47:26.000Z
2018-05-01,6873,40,2018-05-01T00:02:11.000Z,2018-05-31T23:51:24.000Z
2018-06-01,6167,22,2018-06-01T00:39:55.000Z,2018-06-30T23:59:49.000Z
2018-07-01,6292,59,2018-07-01T00:25:07.000Z,2018-07-31T23:54:20.000Z
2018-08-01,6512,91,2018-08-01T00:02:17.000Z,2018-08-31T16:13:44.000Z
2018-09-01,16,15,2018-09-03T09:06:57.000Z,2018-09-29T09:13:03.000Z
2018-10-01,4,4,2018-10-01T15:30:09.000Z,2018-10-17T17:30:18.000Z


In [0]:
status_audit = (orders.groupBy("order_status").agg(F.count("*").alias("order_count"), F.sum(F.col("order_approved_at").isNull().cast("int")).alias("missing_approved_at")).orderBy(F.desc("order_count")))
display(status_audit)

order_status,order_count,missing_approved_at
delivered,96478,14
shipped,1107,0
canceled,625,141
unavailable,609,0
invoiced,314,0
processing,301,0
created,5,5
approved,2,0


## Payment coverage audit

Check whether valid orders have payment records and whether payment values are usable for GMV calculation.

In [0]:
payment_by_order = (
    order_payments
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("order_payment_value"),
        F.count("*").alias("payment_row_count"),
        F.max("payment_sequential").alias("max_payment_sequence")
    )
)

In [0]:
payment_coverage_audit = (
    orders
    .join(payment_by_order, "order_id", "left")
    .groupBy("order_status")
    .agg(
        F.count("*").alias("order_count"),
        
        F.sum(
            F.col("order_payment_value").isNull().cast("int")
        ).alias("orders_without_payment"),
        
        F.sum(
            (F.col("order_payment_value") <= 0).cast("int")
        ).alias("non_positive_payment")
    )
    .orderBy(F.desc("order_count"))
)

display(payment_coverage_audit)

order_status,order_count,orders_without_payment,non_positive_payment
delivered,96478,1,0
shipped,1107,0,0
canceled,625,0,3
unavailable,609,0,0
invoiced,314,0,0
processing,301,0,0
created,5,0,0
approved,2,0,0


## Order item coverage audit

Check whether orders have corresponding order-item records required for category and seller analysis.

In [0]:
orders_with_items = (
    order_items
    .select("order_id")
    .distinct()
    .withColumn("has_items", F.lit(1))
)

In [0]:
item_coverage_audit = (
    orders
    .join(
        orders_with_items,
        on="order_id",
        how="left"
    )
    .groupBy("order_status")
    .agg(
        F.count("*").alias("order_count"),
        F.sum(
            F.col("has_items").isNull().cast("int")
        ).alias("orders_without_items")
    )
    .orderBy(F.desc("order_count"))
)

display(item_coverage_audit)

order_status,order_count,orders_without_items
delivered,96478,0
shipped,1107,1
canceled,625,164
unavailable,609,603
invoiced,314,2
processing,301,0
created,5,5
approved,2,0


In [0]:
valid_statuses = [
    "delivered",
    "shipped",
    "invoiced",
    "processing",
    "approved"
]

In [0]:
valid_missing_item_orders = (
    orders
    .join(
        orders_with_items,
        on="order_id",
        how="left"
    )
    .join(
        payment_by_order,
        on="order_id",
        how="left"
    )
    .filter(
        (F.col("order_purchase_timestamp") >= "2017-01-01") &
        (F.col("order_purchase_timestamp") < "2018-09-01")
    )
    .filter(
        F.col("order_status").isin(valid_statuses)
    )
    .filter(
        F.col("has_items").isNull()
    )
    .select(
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_payment_value"
    )
)

display(valid_missing_item_orders)

order_id,order_status,order_purchase_timestamp,order_payment_value


## Dimension key checks

Verify that customer, product, and seller identifiers are unique before relationship joins.

In [0]:
display(
    customers.agg(
        F.count("*").alias("rows"),
        F.countDistinct("customer_id").alias("unique_customer_ids")
    )
)

display(
    products.agg(
        F.count("*").alias("rows"),
        F.countDistinct("product_id").alias("unique_product_ids")
    )
)

display(
    sellers.agg(
        F.count("*").alias("rows"),
        F.countDistinct("seller_id").alias("unique_seller_ids")
    )
)

rows,unique_customer_ids
99441,99441


rows,unique_product_ids
32951,32951


rows,unique_seller_ids
3095,3095


## Dimension relationship coverage

Check whether customer, product, and seller identifiers in the analysis population have matching dimension records.

In [0]:
analysis_orders_audit = (
    orders
    .join(
        payment_by_order,
        on="order_id",
        how="left"
    )
    .filter(
        (F.col("order_purchase_timestamp") >= "2017-01-01") &
        (F.col("order_purchase_timestamp") < "2018-09-01")
    )
    .filter(
        F.col("order_status").isin(valid_statuses)
    )
    .filter(
        F.col("order_payment_value") > 0
    )
)

In [0]:
customer_lookup = (
    customers
    .select(
        "customer_id",
        "customer_state"
    )
    .withColumn(
        "customer_match",
        F.lit(1)
    )
)

customer_coverage_audit = (
    analysis_orders_audit
    .join(
        customer_lookup,
        on="customer_id",
        how="left"
    )
    .agg(
        F.count("*").alias("analysis_orders"),
        F.sum(
            F.col("customer_match").isNull().cast("int")
        ).alias("orders_without_customer"),
        F.sum(
            F.col("customer_state").isNull().cast("int")
        ).alias("orders_without_customer_state")
    )
)

display(customer_coverage_audit)

analysis_orders,orders_without_customer,orders_without_customer_state
97905,0,0


In [0]:
analysis_items_audit = (
    order_items
    .join(
        analysis_orders_audit.select("order_id"),
        on="order_id",
        how="inner"
    )
)

In [0]:
product_lookup = (
    products
    .select(
        "product_id",
        "product_category_name"
    )
    .withColumn(
        "product_match",
        F.lit(1)
    )
)

seller_lookup = (
    sellers
    .select("seller_id")
    .withColumn(
        "seller_match",
        F.lit(1)
    )
)

In [0]:
item_dimension_coverage_audit = (
    analysis_items_audit
    .join(
        product_lookup,
        on="product_id",
        how="left"
    )
    .join(
        seller_lookup,
        on="seller_id",
        how="left"
    )
    .agg(
        F.count("*").alias("analysis_item_rows"),
        
        F.sum(
            F.col("product_id").isNull().cast("int")
        ).alias("missing_product_id"),
        
        F.sum(
            F.col("product_match").isNull().cast("int")
        ).alias("items_without_product_match"),
        
        F.sum(
            F.col("product_category_name").isNull().cast("int")
        ).alias("items_without_category"),
        
        F.sum(
            F.col("seller_id").isNull().cast("int")
        ).alias("missing_seller_id"),
        
        F.sum(
            F.col("seller_match").isNull().cast("int")
        ).alias("items_without_seller_match"),
        
        F.sum(
            (
                F.col("price").isNull() |
                (F.col("price") <= 0)
            ).cast("int")
        ).alias("invalid_price"),
        
        F.sum(
            (
                F.col("freight_value").isNull() |
                (F.col("freight_value") < 0)
            ).cast("int")
        ).alias("invalid_freight")
    )
)

display(item_dimension_coverage_audit)

analysis_item_rows,missing_product_id,items_without_product_match,items_without_category,missing_seller_id,items_without_seller_match,invalid_price,invalid_freight
111752,0,0,1587,0,0,0,0


In [0]:
display(
    item_dimension_coverage_audit.select(
        "invalid_freight"
    )
)

invalid_freight
0


In [0]:
category_missing_impact = (
    analysis_items_audit
    .join(
        product_lookup,
        on="product_id",
        how="left"
    )
    .agg(
        F.count("*").alias("total_item_count"),
        
        F.sum(
            F.col("product_category_name").isNull().cast("int")
        ).alias("missing_category_item_count"),
        
        F.sum("price").alias("total_merchandise_value"),
        
        F.sum(
            F.when(
                F.col("product_category_name").isNull(),
                F.col("price")
            ).otherwise(0)
        ).alias("missing_category_merchandise_value")
    )
    .withColumn(
        "missing_category_item_pct",
        F.round(
            F.col("missing_category_item_count") /
            F.col("total_item_count") * 100,
            2
        )
    )
    .withColumn(
        "missing_category_value_pct",
        F.round(
            F.col("missing_category_merchandise_value") /
            F.col("total_merchandise_value") * 100,
            2
        )
    )
)

display(category_missing_impact)

total_item_count,missing_category_item_count,total_merchandise_value,missing_category_merchandise_value,missing_category_item_pct,missing_category_value_pct
111752,1587,1.344952967999872E7,178506.66000000027,1.42,1.33


## Audit decisions

### Analysis population

- Analysis period: `2017-01-01` inclusive to `2018-09-01` exclusive.
- 2016 is excluded because transaction volume is sparse and discontinuous.
- September and October 2018 are excluded because they are incomplete residual periods.

### Included order statuses

- `delivered`
- `shipped`
- `invoiced`
- `processing`
- `approved`

### Excluded order statuses

- `created`
- `canceled`
- `unavailable`

### Payment rules

- Orders must have a positive `order_payment_value`.
- One delivered order without a payment record exists in the raw data, but it falls outside the selected analysis period.
- Missing `order_approved_at` is not used as an exclusion rule when the order has a valid status and positive payment.

### Relationship and data-quality findings

- Final audit population: 97,905 orders and 111,752 order-item rows.
- Order, order-item, and payment composite keys are unique.
- All analysis orders have matching customer and customer-state records.
- All analysis-period orders have corresponding order-item records.
- All product and seller identifiers match their dimension tables.
- No invalid or missing item prices were found.
- 1,587 item rows have a missing product category.
- Missing-category items represent 1.42% of item rows and 1.33% of merchandise value.
- Missing product categories will be retained as the `unknown` segment.